In [3]:
import os
import sys
from datetime import datetime
import oracledb

# Initialize Oracle Client in Thick mode for EBS compatibility
INSTANT_CLIENT_DIR = r"C:\Users\E1445612\oracle64client"

try:
    oracledb.init_oracle_client(lib_dir=INSTANT_CLIENT_DIR)
    print(f"[+] Oracle Client initialized in Thick mode from: {INSTANT_CLIENT_DIR}")
except Exception as e:
    print(f"[-] Failed to initialize Oracle Client: {e}")
    sys.exit(1)

# Database connection credentials
DB_USER = "apps"
DB_PASS = "Sukh#na7"
DB_DSN = "BU2EBS.EMRSN.COM:34101/BETSYU2"


def run_gl_period_datafix_generator():
    print("\n" + "=" * 70)
    print(" ORACLE EBS SLA AUTOMATED DATAFIX GENERATOR (GL PERIOD FIX)")
    print("=" * 70)

    # Step 1: Prompt User Directly for Request ID
    request_id = input("\nEnter Accounting Request ID: ").strip()
    if not request_id:
        print("[-] Error: Request ID cannot be empty.")
        return

    print(f"\n[+] Connecting to Oracle EBS Database...")
    try:
        connection = oracledb.connect(user=DB_USER, password=DB_PASS, dsn=DB_DSN)
        cursor = connection.cursor()
        print("[+] Connected successfully to EBS Instance.")
    except Exception as exc:
        print(f"[-] Database Connection Failed: {exc}")
        return

    # Step 2: Extract explicit EVENT_IDs & AE_HEADER_IDs strictly for GL Period Errors
    discovery_query = """
    SELECT DISTINCT 
           xae.event_id,
           xae.ae_header_id,
           xae.ledger_id,
           xae.application_id
      FROM apps.xla_accounting_errors xae
     WHERE xae.request_id = :req_id
       AND (
            xae.encoded_msg LIKE '%95318%' 
         OR xae.encoded_msg LIKE '%95303%' 
         OR LOWER(xae.encoded_msg) LIKE '%period%'
         OR LOWER(xae.encoded_msg) LIKE '%open or a future%'
       )
    """

    print(f"[+] Fetching GL Period errors for Request ID: {request_id}...")
    cursor.execute(discovery_query, req_id=request_id)
    rows = cursor.fetchall()

    if not rows:
        print(f"[-] No GL Period Closed errors found in SLA for Request ID {request_id}.")
        print("    (Note: If this Request ID failed due to CCID/setup issues, they are ignored.)")
        cursor.close()
        connection.close()
        return

    # Consolidate unique IDs
    event_ids = sorted(list(set([r[0] for r in rows if r[0] is not None])))
    header_ids = sorted(list(set([r[1] for r in rows if r[1] is not None])))
    ledger_id = rows[0][2]
    app_id = rows[0][3]

    print(f"\n[+] Isolated Transaction Metadata:")
    print(f"    Request ID      : {request_id}")
    print(f"    Ledger ID       : {ledger_id}")
    print(f"    Application ID  : {app_id}")
    print(f"    Affected Events : {len(event_ids)}")
    print(f"    Affected Headers: {len(header_ids)}")

    # Step 3: Auto-discover Next Open GL Period & Dates from gl_period_statuses
    open_period_query = """
    SELECT period_name, start_date, end_date
      FROM apps.gl_period_statuses
     WHERE ledger_id = :ledger_id
       AND application_id = 101
       AND closing_status = 'O'
       AND start_date >= TRUNC(SYSDATE, 'MM')
     ORDER BY start_date ASC
    """
    
    cursor.execute(open_period_query, ledger_id=ledger_id)
    period_row = cursor.fetchone()

    if period_row:
        target_period, open_start_date, open_end_date = period_row
        target_date = open_start_date.strftime('%d-%b-%Y').upper()
        ref_date = open_end_date.strftime('%d-%b-%Y').upper()
        print(f"\n[+] Querying gl_period_statuses for Ledger {ledger_id}...")
        print(f"    Target Open GL Period : {target_period}")
        print(f"    Target Accounting Date: {target_date}")
        print(f"    Target Reference Date : {ref_date}")
    else:
        print(f"\n[-] Warning: No Open GL Period found for Ledger {ledger_id} in gl_period_statuses.")
        cursor.close()
        connection.close()
        return

    cursor.close()
    connection.close()

    event_ids_str = ", ".join(map(str, event_ids))
    header_ids_str = ", ".join(map(str, header_ids)) if header_ids else event_ids_str

    # Step 4: Construct 3 SQL Update Statements Template
    sql_template = f"""-- =================================================================================
-- SLA AUTOMATED GL PERIOD CLOSED DATAFIX
-- Request ID     : {request_id}
-- Application ID : {app_id}
-- Ledger ID      : {ledger_id}
-- Target Date    : {target_date}
-- Target Ref Date: {ref_date}
-- Period Name    : {target_period}
-- Total Events   : {len(event_ids)}
-- Generated On   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
-- =================================================================================

-- ---------------------------------------------------------------------------------
-- STEP 1: UPDATE XLA_EVENTS (EVENT DATE & REFERENCE DATE 1)
-- ---------------------------------------------------------------------------------
UPDATE apps.xla_events 
   SET event_date = TO_DATE('{target_date}', 'DD-MON-YYYY'), 
       reference_date_1 = TO_DATE('{ref_date}', 'DD-MON-YYYY')
 WHERE event_id IN ({event_ids_str});

-- ---------------------------------------------------------------------------------
-- STEP 2: UPDATE XLA_AE_HEADERS (ACCOUNTING DATE & PERIOD NAME)
-- ---------------------------------------------------------------------------------
UPDATE apps.xla_ae_headers 
   SET accounting_date = TO_DATE('{target_date}', 'DD-MON-YYYY'), 
       period_name = '{target_period}'
 WHERE event_id IN ({event_ids_str});

-- ---------------------------------------------------------------------------------
-- STEP 3: UPDATE XLA_AE_LINES (ACCOUNTING DATE)
-- ---------------------------------------------------------------------------------
UPDATE apps.xla_ae_lines 
   SET accounting_date = TO_DATE('{target_date}', 'DD-MON-YYYY')
 WHERE ae_header_id IN ({header_ids_str});

-- COMMIT;
"""

    output_filename = f"sla_gl_period_fix_req_{request_id}.sql"
    with open(output_filename, "w") as file:
        file.write(sql_template)

    print("\n" + "=" * 70)
    print(" SUCCESS: Datafix SQL Script Successfully Generated!")
    print("=" * 70)
    print(f" Output File   : {os.path.abspath(output_filename)}")
    print(f" Request ID    : {request_id}")
    print(f" Target Date   : {target_date}")
    print(f" Period Name   : {target_period}")
    print(f" Events Fixed  : {len(event_ids)}")
    print("=" * 70 + "\n")


if __name__ == "__main__":
    run_gl_period_datafix_generator()

[+] Oracle Client initialized in Thick mode from: C:\Users\E1445612\oracle64client

 ORACLE EBS SLA AUTOMATED DATAFIX GENERATOR (GL PERIOD FIX)

[+] Connecting to Oracle EBS Database...
[+] Connected successfully to EBS Instance.
[+] Fetching GL Period errors for Request ID: 345998756...
[-] No GL Period Closed errors found in SLA for Request ID 345998756.
    (Note: If this Request ID failed due to CCID/setup issues, they are ignored.)
